In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q22/q22_rewrite/checkpoints/pre_cell_2.pickle

In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Select necessary columns and compute country code
customer_filtered = customer[['C_ACCTBAL', 'C_CUSTKEY']].assign(
    CNTRYCODE=customer['C_PHONE'].str.slice(0, 2)
)

# Filter by positive balance and desired country codes
valid_codes = ['13', '31', '23', '29', '30', '18', '17']
mask = (
    (customer_filtered['C_ACCTBAL'] > 0) &
    customer_filtered['CNTRYCODE'].isin(valid_codes)
)
customer_filtered = customer_filtered[mask]

# Compute average balance and filter above average
avg_value = customer_filtered['C_ACCTBAL'].mean()
customer_filtered = customer_filtered[customer_filtered['C_ACCTBAL'] > avg_value]

# Exclude customers who have placed orders using a GPU-friendly isin filter
customer_selected = customer_filtered[~customer_filtered['C_CUSTKEY'].isin(orders['O_CUSTKEY'])]

# Keep only the columns needed for aggregation
customer_selected = customer_selected[['CNTRYCODE', 'C_ACCTBAL']]

# Compute number of customers per country (GPU groupby.size)
agg1 = (
    customer_selected
    .groupby('CNTRYCODE', as_index=False)
    .size()
    .rename(columns={'size': 'NUMCUST'})
)

# Compute total account balance per country (GPU groupby.sum)
agg2 = (
    customer_selected
    .groupby('CNTRYCODE', as_index=False)['C_ACCTBAL']
    .sum()
    .rename(columns={'C_ACCTBAL': 'TOTACCTBAL'})
)

# Join the two aggregations and sort by country code
total = agg1.merge(agg2, on='CNTRYCODE').sort_values('CNTRYCODE')

CPU times: user 85.6 ms, sys: 31.5 ms, total: 117 ms
Wall time: 117 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q22/rewritten/o4_mini_high_small/checkpoints/post_cell_2_try_4.pickle